# NLP Robustness Study — SST-2
**Team:** Curtis Cao, Xilu Zeng, Soham Agarwal, Ian Abusamra

This notebook is the Colab entry point. It installs dependencies, downloads GloVe, then runs the full experiment pipeline.

---
**Sections**
1. Setup (install deps, mount Drive)
2. Download GloVe embeddings
3. Verify perturbations (sanity check)
4. Train baseline models
5. Run full experiment sweep
6. Analyze & visualize results

## 1. Setup

In [ ]:
!pip install -q datasets transformers torch scikit-learn seaborn tqdm joblib

In [ ]:
# Mount Google Drive (skip if using a university cluster)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Option A: project already in Drive
PROJECT_ROOT = '/content/drive/MyDrive/NLPProject'  # <-- adjust if needed

# Option B: clone from GitHub
# !git clone https://github.com/<your-repo>/NLPProject.git /content/NLPProject
# PROJECT_ROOT = '/content/NLPProject'

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

## 2. Download GloVe Embeddings
Only needed for the NBOW model. Skipped on subsequent runs if the file already exists.

In [ ]:
import os

GLOVE_DIR = os.path.join(PROJECT_ROOT, 'data')
GLOVE_FILE = os.path.join(GLOVE_DIR, 'glove.6B.100d.txt')

if not os.path.exists(GLOVE_FILE):
    print('Downloading GloVe 6B 100d (~330 MB) ...')
    os.makedirs(GLOVE_DIR, exist_ok=True)
    import subprocess
    subprocess.run(['wget', '-q', '--show-progress',
                    '-O', '/tmp/glove.6B.zip',
                    'https://nlp.stanford.edu/data/glove.6B.zip'], check=True)
    subprocess.run(['unzip', '-q', '/tmp/glove.6B.zip',
                    'glove.6B.100d.txt', '-d', GLOVE_DIR], check=True)
    print('Done.')
else:
    print(f'GloVe already exists at {GLOVE_FILE}')

## 3. Verify Perturbations
Inspect example outputs at each severity level before running experiments.

In [ ]:
!python data/perturbations.py

## 4. Train Baseline Models
Trains LR and NBOW on the SST-2 training split; caches models to `results/`.
DistilBERT uses a pre-fine-tuned checkpoint — no training needed.

In [ ]:
# Logistic Regression + TF-IDF (~30s)
!python models/baseline_lr.py

In [ ]:
# NBOW: GloVe load + 10 epochs MLP (~5-10 min on GPU)
!python models/baseline_nbow.py

In [ ]:
# DistilBERT: download checkpoint + verify clean accuracy
!python models/bert_eval.py

## 5. Run Full Experiment Sweep
Evaluates 3 models × 13 conditions → `results/results.csv` (39 rows).
Expected runtime on Colab GPU: ~20–30 minutes.

In [ ]:
!python experiments/run_experiments.py

In [ ]:
# Preview the results table
import pandas as pd
df = pd.read_csv('results/results.csv')
print(df.to_string(index=False))

## 6. Analyze & Visualize Results

In [ ]:
!python experiments/analyze_results.py

In [ ]:
from IPython.display import Image, display
display(Image('results/plots/accuracy_curves.png'))
display(Image('results/plots/drop_heatmap.png'))